# Hidden stability hyperplane in transformed BvK force-constant space

This notebook determines the explicit approximate hyperplane associated with the nearly-null fifth PCA direction.

The transformed feature vector is

\[
\mathbf{x}=
\left(
\alpha_0,
\alpha_1+2\beta_1,
\alpha_1-\beta_1,
\alpha_2,
\beta_2
\right).
\]

The workflow is:

1. Load the four MOGA dataframe files.
2. Select the top solutions per \((m,a)\) for each dataframe.
3. Compute the transformed force-constant coordinates.
4. Fit PCA to the standardized feature matrix.
5. Extract the fifth principal component and convert it into an explicit hyperplane equation.
6. Compare the PCA-null coordinate with an independent total-least-squares hyperplane fit.
7. Generate publication-quality figures and CSV summary files.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -----------------------------
# User settings
# -----------------------------
PKL_DIR = Path("./dataframes")
OUTPUT_DIR = Path("./hidden_stability_hyperplane_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATAFRAME_FILES = {
    "dataframe000013": PKL_DIR / "dataframe000013.pkl",
    "dataframe000014": PKL_DIR / "dataframe000014.pkl",
    "dataframe000015": PKL_DIR / "dataframe000015.pkl",
    "dataframe000016": PKL_DIR / "dataframe000016.pkl",
}

TOP_K = 5
SORT_COLUMN = "fitness_norm"
ASCENDING = False

# If True, fit PCA using only dynamically stable rows when available.
# If False, use the selected top-K rows exactly as in the earlier PCA notebooks.
USE_STABLE_ONLY_FOR_FIT = False

# Optional stability filter used only if USE_STABLE_ONLY_FOR_FIT=True
MIN_FREQUENCY_TOL = -1e-8

# Publication-quality plotting defaults
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 15,
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

print(f"Output directory: {OUTPUT_DIR.resolve()}")


## Load data and select top solutions

In [ ]:
def load_one_dataframe(label, path):
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Update PKL_DIR or DATAFRAME_FILES near the top of the notebook."
        )
    df = pd.read_pickle(path).copy()
    df["dataset"] = label
    return df

frames = []
for label, path in DATAFRAME_FILES.items():
    df_i = load_one_dataframe(label, path)
    print(f"{label}: {len(df_i):,} rows")
    frames.append(df_i)

df_all = pd.concat(frames, ignore_index=True)
print(f"Total rows loaded: {len(df_all):,}")

required_columns = [
    "dataset", "mass", "a_val", "alpha0", "alpha1", "beta1", "alpha2", "beta2", SORT_COLUMN
]
missing = [c for c in required_columns if c not in df_all.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

df_top = (
    df_all.sort_values(SORT_COLUMN, ascending=ASCENDING)
          .groupby(["dataset", "mass", "a_val"], as_index=False, group_keys=False)
          .head(TOP_K)
          .reset_index(drop=True)
)

print(f"Top-{TOP_K} rows selected: {len(df_top):,}")
print(df_top[["dataset", "mass", "a_val", SORT_COLUMN]].head())


## Derived variables and family labels

In [ ]:
df = df_top.copy()

df["alpha1_plus_2beta1"] = df["alpha1"] + 2.0 * df["beta1"]
df["alpha1_minus_beta1"] = df["alpha1"] - df["beta1"]

den = df["alpha2"].abs() + df["beta2"].abs()
df["r_alpha2"] = np.where(den > 0, df["alpha2"].abs() / den, np.nan)

def classify_second_neighbor_family(r):
    if pd.isna(r):
        return "undefined"
    if r >= 0.80:
        return r"$\alpha_2$-dominated"
    if r <= 0.20:
        return r"$\beta_2$-dominated"
    return "mixed"

df["second_neighbor_family"] = df["r_alpha2"].map(classify_second_neighbor_family)

FEATURES = [
    "alpha0",
    "alpha1_plus_2beta1",
    "alpha1_minus_beta1",
    "alpha2",
    "beta2",
]

DISPLAY_LABELS = {
    "alpha0": r"$\alpha_0$",
    "alpha1_plus_2beta1": r"$\alpha_1 + 2\beta_1$",
    "alpha1_minus_beta1": r"$\alpha_1 - \beta_1$",
    "alpha2": r"$\alpha_2$",
    "beta2": r"$\beta_2$",
}

if USE_STABLE_ONLY_FOR_FIT and "min_frequency" in df.columns:
    fit_mask = df["min_frequency"] >= MIN_FREQUENCY_TOL
else:
    fit_mask = np.ones(len(df), dtype=bool)

df_fit = df.loc[fit_mask].copy()
print(f"Rows used for PCA/hyperplane fit: {len(df_fit):,}")
print(df[FEATURES + ["r_alpha2", "second_neighbor_family"]].head())


## PCA and the nearly-null hyperplane

PCA is applied to the standardized matrix

\[
z_j = \frac{x_j-\mu_j}{\sigma_j}.
\]

The fifth PCA direction is the normal vector of the nearly-null hyperplane in standardized coordinates:

\[
C = \sum_j w_j z_j \approx 0.
\]

The same equation can be written in raw force-constant coordinates as

\[
\sum_j a_j x_j + b \approx 0,
\]

where

\[
a_j = \frac{w_j}{\sigma_j},
\qquad
b = -\sum_j \frac{w_j \mu_j}{\sigma_j}.
\]


In [ ]:
X_fit = df_fit[FEATURES].to_numpy(dtype=float)

scaler = StandardScaler()
Z_fit = scaler.fit_transform(X_fit)

pca = PCA(n_components=len(FEATURES), random_state=0)
S_fit = pca.fit_transform(Z_fit)

explained = pca.explained_variance_ratio_
loadings = pca.components_.T
pc5_w = pca.components_[-1].copy()

# Choose a deterministic sign convention: make the sum of loadings positive.
if pc5_w.sum() < 0:
    pc5_w *= -1
    pca.components_[-1, :] *= -1
    S_fit[:, -1] *= -1

# Apply same scaler/PCA to all selected rows.
Z_all = scaler.transform(df[FEATURES].to_numpy(dtype=float))
S_all = Z_all @ pca.components_.T
C = Z_all @ pc5_w
df["C_pc5"] = C
for i in range(len(FEATURES)):
    df[f"PC{i+1}"] = S_all[:, i]

raw_coeff = pc5_w / scaler.scale_
raw_intercept = -np.sum(pc5_w * scaler.mean_ / scaler.scale_)
df["hyperplane_residual_raw"] = df[FEATURES].to_numpy(dtype=float) @ raw_coeff + raw_intercept

summary = pd.DataFrame({
    "feature": FEATURES,
    "display_label": [DISPLAY_LABELS[f] for f in FEATURES],
    "mean_raw": scaler.mean_,
    "scale_raw": scaler.scale_,
    "pc5_loading_standardized": pc5_w,
    "hyperplane_coeff_raw": raw_coeff,
})

print("Explained variance ratio:")
for i, ev in enumerate(explained, start=1):
    print(f"PC{i}: {ev:.12e}")

print("\nHyperplane in standardized coordinates:")
terms_std = [f"({w:+.8f}) z[{DISPLAY_LABELS[f]}]" for f, w in zip(FEATURES, pc5_w)]
print("C = " + " ".join(terms_std) + " ≈ 0")

print("\nHyperplane in raw coordinates:")
terms_raw = [f"({a:+.8e}) {DISPLAY_LABELS[f]}" for f, a in zip(FEATURES, raw_coeff)]
print(" ".join(terms_raw) + f" ({raw_intercept:+.8e}) ≈ 0")

print("\nSummary table:")
display(summary)

summary.to_csv(OUTPUT_DIR / "pc5_hyperplane_coefficients.csv", index=False)
df.to_csv(OUTPUT_DIR / "top5_with_constraint_coordinate_C.csv", index=False)


## Independent total-least-squares hyperplane fit

As a check, we fit the minimum-variance hyperplane directly by singular value decomposition of the standardized data. This should recover the same normal vector as PC5, up to a sign.


In [ ]:
# Centered standardized data already has zero mean in the fit set.
_, singular_values, vh = np.linalg.svd(Z_fit, full_matrices=False)
tls_w = vh[-1, :].copy()
if np.dot(tls_w, pc5_w) < 0:
    tls_w *= -1

cosine_similarity = np.dot(tls_w, pc5_w) / (np.linalg.norm(tls_w) * np.linalg.norm(pc5_w))
angle_deg = np.degrees(np.arccos(np.clip(cosine_similarity, -1, 1)))

print("Singular values of standardized feature matrix:")
for i, sv in enumerate(singular_values, start=1):
    print(f"s{i}: {sv:.12e}")
print(f"\nCosine similarity between PC5 and TLS normals: {cosine_similarity:.12f}")
print(f"Angle between normals: {angle_deg:.6e} degrees")

comparison = pd.DataFrame({
    "feature": FEATURES,
    "display_label": [DISPLAY_LABELS[f] for f in FEATURES],
    "pc5_normal": pc5_w,
    "tls_normal": tls_w,
    "difference": pc5_w - tls_w,
})
display(comparison)
comparison.to_csv(OUTPUT_DIR / "pc5_vs_tls_normal_comparison.csv", index=False)


## Numerical spread of the constraint coordinate

In [ ]:
stats = df["C_pc5"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print(stats)
print("\nRMS(C):", np.sqrt(np.mean(df["C_pc5"]**2)))
print("Mean |C|:", np.mean(np.abs(df["C_pc5"])))
print("Max |C|:", np.max(np.abs(df["C_pc5"])))


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.6))
ax.hist(df["C_pc5"], bins=40, edgecolor="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=1.2)
ax.set_xlabel(r"$C$")
ax.set_ylabel("Count")
ax.set_title(r"Spread of the hyperplane residual $C$")
fig.savefig(OUTPUT_DIR / "constraint_coordinate_C_histogram.pdf")
fig.savefig(OUTPUT_DIR / "constraint_coordinate_C_histogram.png")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 5.6))
abs_sorted = np.sort(np.abs(df["C_pc5"].to_numpy()))
ax.plot(np.arange(len(abs_sorted)), abs_sorted, marker="o", linestyle="none", markersize=4)
ax.set_yscale("log")
ax.set_xlabel("Sorted sample index")
ax.set_ylabel(r"$|C|$")
ax.set_title(r"Sorted absolute hyperplane residual")
fig.savefig(OUTPUT_DIR / "constraint_coordinate_C_sorted_abs.pdf")
fig.savefig(OUTPUT_DIR / "constraint_coordinate_C_sorted_abs.png")
plt.show()


## Relation to fitness metrics and phonon stability

In [ ]:
def scatter_C_vs(column, xlabel=None, filename=None, color_by="r_alpha2"):
    if column not in df.columns:
        print(f"Skipping {column}: column not found")
        return
    fig, ax = plt.subplots(figsize=(7.2, 5.6))
    sc = ax.scatter(
        df[column], df["C_pc5"],
        c=df[color_by] if color_by in df.columns else None,
        s=52, alpha=0.85, edgecolors="black", linewidths=0.45
    )
    ax.axhline(0, color="black", linewidth=1.2)
    ax.set_xlabel(xlabel or column)
    ax.set_ylabel(r"$C$")
    ax.set_title(rf"Constraint coordinate versus {xlabel or column}")
    if color_by in df.columns:
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label(r"$r_{\alpha_2}$")
    if filename is None:
        safe = re.sub(r"[^a-zA-Z0-9_]+", "_", column)
        filename = f"constraint_coordinate_C_vs_{safe}"
    fig.savefig(OUTPUT_DIR / f"{filename}.pdf")
    fig.savefig(OUTPUT_DIR / f"{filename}.png")
    plt.show()

scatter_C_vs("fitness_norm", xlabel=r"$f_{\rm norm}$", filename="constraint_coordinate_C_vs_fitness_norm")
scatter_C_vs("fitness1", xlabel="fitness1", filename="constraint_coordinate_C_vs_fitness1")
scatter_C_vs("fitness2", xlabel="fitness2", filename="constraint_coordinate_C_vs_fitness2")
scatter_C_vs("fitness3", xlabel="fitness3", filename="constraint_coordinate_C_vs_fitness3")
scatter_C_vs("min_frequency", xlabel=r"$\omega_{\min}$ (THz)", filename="constraint_coordinate_C_vs_min_frequency")
scatter_C_vs("max_frequency", xlabel=r"$\omega_{\max}$ (THz)", filename="constraint_coordinate_C_vs_max_frequency")
scatter_C_vs("num_imaginary", xlabel=r"$N_-$", filename="constraint_coordinate_C_vs_num_imaginary")


## PCA map colored by the hyperplane residual

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.2))
sc = ax.scatter(
    df["PC1"], df["PC2"], c=df["C_pc5"], s=48,
    alpha=0.9, edgecolors="black", linewidths=0.35
)
ax.axhline(0, color="0.75", linewidth=1.2)
ax.axvline(0, color="0.75", linewidth=1.2)
ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
ax.set_title(r"PCA map colored by hyperplane residual $C$")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"$C$")
fig.savefig(OUTPUT_DIR / "pca_map_colored_by_constraint_coordinate_C.pdf")
fig.savefig(OUTPUT_DIR / "pca_map_colored_by_constraint_coordinate_C.png")
plt.show()


## Constraint coordinate by second-neighbor family

In [ ]:
family_order = [r"$\beta_2$-dominated", "mixed", r"$\alpha_2$-dominated"]
fig, ax = plt.subplots(figsize=(8.5, 5.8))
positions = np.arange(1, len(family_order) + 1)

box_data = [df.loc[df["second_neighbor_family"] == fam, "C_pc5"].dropna().to_numpy() for fam in family_order]
ax.boxplot(box_data, positions=positions, widths=0.55, showfliers=True)

rng = np.random.default_rng(7)
for pos, fam, vals in zip(positions, family_order, box_data):
    if len(vals) == 0:
        continue
    jitter = rng.normal(0, 0.045, size=len(vals))
    ax.scatter(np.full(len(vals), pos) + jitter, vals, s=28, alpha=0.45)

ax.axhline(0, color="black", linewidth=1.2)
ax.set_xticks(positions)
ax.set_xticklabels(family_order)
ax.set_ylabel(r"$C$")
ax.set_title(r"Hyperplane residual by second-neighbor family")
fig.savefig(OUTPUT_DIR / "constraint_coordinate_C_by_second_neighbor_family.pdf")
fig.savefig(OUTPUT_DIR / "constraint_coordinate_C_by_second_neighbor_family.png")
plt.show()

family_stats = (
    df.groupby("second_neighbor_family")["C_pc5"]
      .agg(["count", "mean", "std", "median", "min", "max"])
      .reindex(family_order)
)
display(family_stats)
family_stats.to_csv(OUTPUT_DIR / "constraint_coordinate_C_by_family_stats.csv")


## Optional: raw-coordinate LaTeX equation

This cell prints a manuscript-ready version of the fitted hyperplane. The equation uses the raw force-constant units of the dataframe.


In [ ]:
def latex_raw_hyperplane(coeff, intercept, labels):
    parts = []
    for a, lab in zip(coeff, labels):
        sign = "+" if a >= 0 else "-"
        parts.append(f" {sign} {abs(a):.6g}{lab}")
    sign_b = "+" if intercept >= 0 else "-"
    eq = "".join(parts).lstrip(" +") + f" {sign_b} {abs(intercept):.6g} \\simeq 0"
    return eq

labels_latex = [
    r"\alpha_0",
    r"(\alpha_1+2\beta_1)",
    r"(\alpha_1-\beta_1)",
    r"\alpha_2",
    r"\beta_2",
]

eq = latex_raw_hyperplane(raw_coeff, raw_intercept, labels_latex)
print("Raw-coordinate hyperplane equation:")
print("\\[")
print(eq)
print("\\]")

print("\nStandardized-coordinate equation:")
std_terms = []
for w, lab in zip(pc5_w, labels_latex):
    sign = "+" if w >= 0 else "-"
    std_terms.append(f" {sign} {abs(w):.6g} z_{{{lab}}}")
print("\\[")
print("C = " + "".join(std_terms).lstrip(" +") + r" \simeq 0")
print("\\]")


## Save final compact tables

In [ ]:
compact_cols = [
    "dataset", "mass", "a_val", "alpha0", "alpha1", "beta1", "alpha2", "beta2",
    "alpha1_plus_2beta1", "alpha1_minus_beta1", "r_alpha2", "second_neighbor_family",
    "C_pc5", "hyperplane_residual_raw", "PC1", "PC2", "PC3", "PC4", "PC5",
]
compact_cols = [c for c in compact_cols if c in df.columns]
compact = df[compact_cols].copy()
compact.to_csv(OUTPUT_DIR / "compact_top5_constraint_hyperplane_results.csv", index=False)
print("Saved:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path)
